In [ ]:
# Cell 1: Environment Setup
!pip install pandas numpy scikit-learn matplotlib pyarrow -q

import pandas as pd
import numpy as np
import os, json, csv, time, random, logging
from datetime import date, datetime, timedelta

random.seed(42)
np.random.seed(42)
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
print("✓ Environment ready for: OLTP vs OLAP and Data Storage Systems")

✓ Environment ready for: OLTP vs OLAP and Data Storage Systems



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\9901359\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
# Cell 2: Generate Dataset
PRODUCTS = ['Laptop','Monitor','Keyboard','Mouse','Headset','Webcam','USB Hub','Desk Lamp','Speaker','Tablet']
REGIONS = ['North','South','East','West']
STATUSES = ['completed','pending','cancelled','refunded']

records = []
for i in range(10000):
    records.append({
        'order_id': f'ORD-{i:05d}',
        'product': random.choice(PRODUCTS),
        'category': PRODUCTS[i % len(PRODUCTS)].split()[0] if i < 50 else random.choice(['Electronics','Accessories','Home','Sports','Office']),
        'region': random.choice(REGIONS),
        'quantity': random.randint(1, 50),
        'unit_price': round(random.uniform(9.99, 999.99), 2),
        'order_date': str(date(2024,1,1) + timedelta(days=random.randint(0,364))),
        'status': random.choices(STATUSES, weights=[70,15,10,5])[0]
    })

df = pd.DataFrame(records)
df['revenue'] = (df['quantity'] * df['unit_price']).round(2)

print(f"✓ Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

✓ Dataset loaded: 10000 rows, 9 columns


,order_id,product,category,region,quantity,unit_price,order_date,status,revenue
0,ORD-00000,Monitor,Laptop,North,48,282.27,2024-04-24,completed,13548.96
1,ORD-00001,Monitor,Monitor,North,38,427.69,2024-01-16,completed,16252.22
2,ORD-00002,Mouse,Keyboard,North,36,206.84,2024-11-28,pending,7446.24
3,ORD-00003,USB Hub,Mouse,South,29,593.36,2024-01-04,pending,17207.44
4,ORD-00004,Keyboard,Headset,West,22,285.08,2024-04-20,refunded,6271.76


In [ ]:
# Cell 3: Data Profiling
print('=' * 60)
print('DATA PROFILE: OLTP vs OLAP and Data Storage Systems')
print('=' * 60)
print(f'\nShape: {df.shape}')
print(f'\nColumn Types:\n{df.dtypes}')
print(f'\nNull Counts:\n{df.isnull().sum()}')
print(f'\nDuplicate Rows: {df.duplicated().sum()}')
print(f'\nNumeric Summary:\n{df.describe()}')
print(f'\nUnique Values:')
for col in df.columns:
    print(f'  {col}: {df[col].nunique()} unique')
print(f'\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

DATA PROFILE: OLTP vs OLAP and Data Storage Systems

Shape: (10000, 9)

Column Types:
order_id          str
product           str
category          str
region            str
quantity        int64
unit_price    float64
order_date        str
status            str
revenue       float64
dtype: object

Null Counts:
order_id      0
product       0
category      0
region        0
quantity      0
unit_price    0
order_date    0
status        0
revenue       0
dtype: int64

Duplicate Rows: 0

Numeric Summary:
           quantity    unit_price       revenue
count  10000.000000  10000.000000  10000.000000
mean      25.140600    503.807532  12629.256019
std       14.580209    287.654131  11033.664281
min        1.000000     10.050000     16.440000
25%       12.000000    254.082500   3529.290000
50%       25.000000    502.315000   9347.180000
75%       38.000000    751.495000  19358.000000
max       50.000000    999.970000  49685.500000

Unique Values:
  order_id: 10000 unique
  product: 10 unique


In [ ]:
# Cell 4: Core Implementation — OLTP vs OLAP and Data Storage Systems

import sqlite3
import random
from datetime import date, timedelta

random.seed(42)

# ============================================================
# PART 1: OLTP Database — Normalized 3NF Schema
# OLTP = Online Transaction Processing (fast inserts/updates)
# ============================================================
oltp_conn = sqlite3.connect(':memory:')
oltp_cur = oltp_conn.cursor()

# Create normalized tables (3NF — no redundancy)
oltp_cur.executescript('''
    CREATE TABLE customers (
        customer_id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        email TEXT UNIQUE,
        region TEXT
    );
    CREATE TABLE products (
        product_id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        category TEXT,
        unit_price REAL
    );
    CREATE TABLE orders (
        order_id INTEGER PRIMARY KEY,
        customer_id INTEGER REFERENCES customers(customer_id),
        order_date TEXT,
        status TEXT DEFAULT 'pending'
    );
    CREATE TABLE order_items (
        item_id INTEGER PRIMARY KEY,
        order_id INTEGER REFERENCES orders(order_id),
        product_id INTEGER REFERENCES products(product_id),
        quantity INTEGER,
        line_total REAL
    );
''')

# Populate OLTP tables with sample data
REGIONS = ['North', 'South', 'East', 'West']
PRODUCTS = ['Laptop', 'Monitor', 'Keyboard', 'Mouse', 'Headset',
            'Webcam', 'USB Hub', 'Desk Lamp', 'Speaker', 'Tablet']
CATEGORIES = ['Electronics', 'Accessories', 'Home', 'Office']

for i in range(1, 51):
    oltp_cur.execute('INSERT INTO customers VALUES (?, ?, ?, ?)',
                     (i, f'Customer_{i}', f'cust{i}@datamart.com', random.choice(REGIONS)))

for i, prod in enumerate(PRODUCTS, 1):
    price = round(random.uniform(19.99, 999.99), 2)
    oltp_cur.execute('INSERT INTO products VALUES (?, ?, ?, ?)',
                     (i, prod, random.choice(CATEGORIES), price))

for i in range(1, 101):
    cust_id = random.randint(1, 50)
    order_date = str(date(2024, 1, 1) + timedelta(days=random.randint(0, 364)))
    status = random.choice(['completed', 'pending', 'cancelled'])
    oltp_cur.execute('INSERT INTO orders VALUES (?, ?, ?, ?)',
                     (i, cust_id, order_date, status))
    # Each order has 1-3 line items
    for j in range(random.randint(1, 3)):
        prod_id = random.randint(1, len(PRODUCTS))
        qty = random.randint(1, 10)
        price = oltp_cur.execute('SELECT unit_price FROM products WHERE product_id=?',
                                 (prod_id,)).fetchone()[0]
        oltp_cur.execute('INSERT INTO order_items (order_id, product_id, quantity, line_total) VALUES (?, ?, ?, ?)',
                         (i, prod_id, qty, round(qty * price, 2)))

oltp_conn.commit()

# OLTP query: single order lookup (fast, uses primary key)
row = oltp_cur.execute('''
    SELECT o.order_id, c.name, o.order_date, o.status,
           COUNT(oi.item_id) AS items, SUM(oi.line_total) AS total
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_id = 1
    GROUP BY o.order_id
''').fetchone()
print(f'OLTP single-order lookup: Order #{row[0]}, Customer: {row[1]}, '
      f'Date: {row[2]}, Status: {row[3]}, Items: {row[4]}, Total: ${row[5]:.2f}')
print('OLTP schema created')

# ============================================================
# PART 2: OLAP Database — Star Schema (Denormalized)
# OLAP = Online Analytical Processing (fast aggregations)
# ============================================================
olap_conn = sqlite3.connect(':memory:')
olap_cur = olap_conn.cursor()

olap_cur.executescript('''
    CREATE TABLE dim_customer (
        customer_key INTEGER PRIMARY KEY,
        customer_id INTEGER,
        name TEXT,
        region TEXT
    );
    CREATE TABLE dim_product (
        product_key INTEGER PRIMARY KEY,
        product_id INTEGER,
        name TEXT,
        category TEXT
    );
    CREATE TABLE dim_date (
        date_key INTEGER PRIMARY KEY,
        full_date TEXT,
        year INTEGER,
        month INTEGER,
        quarter INTEGER,
        day_of_week TEXT
    );
    CREATE TABLE fact_sales (
        sale_key INTEGER PRIMARY KEY AUTOINCREMENT,
        customer_key INTEGER REFERENCES dim_customer(customer_key),
        product_key INTEGER REFERENCES dim_product(product_key),
        date_key INTEGER REFERENCES dim_date(date_key),
        quantity INTEGER,
        revenue REAL,
        status TEXT
    );
''')

# ETL: Load dimensions from OLTP
for row in oltp_cur.execute('SELECT customer_id, name, region FROM customers'):
    olap_cur.execute('INSERT INTO dim_customer VALUES (?, ?, ?, ?)',
                     (row[0], row[0], row[1], row[2]))
for row in oltp_cur.execute('SELECT product_id, name, category FROM products'):
    olap_cur.execute('INSERT INTO dim_product VALUES (?, ?, ?, ?)',
                     (row[0], row[0], row[1], row[2]))

# Generate date dimension for 2024
for day_offset in range(366):
    d = date(2024, 1, 1) + timedelta(days=day_offset)
    date_key = int(d.strftime('%Y%m%d'))
    olap_cur.execute('INSERT INTO dim_date VALUES (?, ?, ?, ?, ?, ?)',
                     (date_key, str(d), d.year, d.month, (d.month - 1) // 3 + 1, d.strftime('%A')))

# Load fact table from OLTP joins
for row in oltp_cur.execute('''
    SELECT o.customer_id, oi.product_id, o.order_date, oi.quantity, oi.line_total, o.status
    FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
'''):
    date_key = int(row[2].replace('-', ''))
    olap_cur.execute('INSERT INTO fact_sales (customer_key, product_key, date_key, quantity, revenue, status) VALUES (?, ?, ?, ?, ?, ?)',
                     (row[0], row[1], date_key, row[3], row[4], row[5]))

olap_conn.commit()

# OLAP query: revenue by region and quarter (fast aggregation)
olap_results = olap_cur.execute('''
    SELECT dc.region, dd.quarter, SUM(fs.revenue) AS total_revenue, COUNT(*) AS txn_count
    FROM fact_sales fs
    JOIN dim_customer dc ON fs.customer_key = dc.customer_key
    JOIN dim_date dd ON fs.date_key = dd.date_key
    WHERE fs.status = 'completed'
    GROUP BY dc.region, dd.quarter
    ORDER BY dc.region, dd.quarter
''').fetchall()

print('\nOLAP star schema created')
print('\nRevenue by Region and Quarter (OLAP):')
print(f'{"Region":<10} {"Q":>3} {"Revenue":>12} {"Txns":>6}')
print('-' * 35)
for row in olap_results:
    print(f'{row[0]:<10} Q{row[1]:>1} ${row[2]:>10,.2f} {row[3]:>6}')

# Compare query patterns
print('\n--- OLTP vs OLAP Comparison ---')
print(f'OLTP tables: 4 (normalized, no redundancy)')
print(f'OLAP tables: 4 (star schema, denormalized for speed)')
oltp_item_count = oltp_cur.execute('SELECT COUNT(*) FROM order_items').fetchone()[0]
olap_fact_count = olap_cur.execute('SELECT COUNT(*) FROM fact_sales').fetchone()[0]
print(f'OLTP order_items rows: {oltp_item_count}')
print(f'OLAP fact_sales rows:  {olap_fact_count}')

oltp_conn.close()
olap_conn.close()
print('\n-- Core implementation complete')

In [ ]:
# Cell 5: Results Analysis
print('=' * 60)
print('RESULTS: OLTP vs OLAP and Data Storage Systems')
print('=' * 60)
print(f'Input rows: {len(df)}')
print(f'Processing complete')

# Display key metrics
print(f'\nRevenue by Region:')
print(df.groupby('region')['revenue'].sum().sort_values(ascending=False))

In [ ]:
# Cell 6: Self-Check — OLTP vs OLAP
# Run this cell to verify your work before clicking "Run Tests"
print('=' * 50)
print('SELF-CHECK — OLTP vs OLAP')
print('=' * 50)

checks = [
    ('len(df) >= 100', lambda: len(df) >= 100),
    ('"product" in df.columns', lambda: "product" in df.columns),
    ('"region" in df.columns', lambda: "region" in df.columns),
    ('df.groupby("region")["revenue"].sum().sum() > 0', lambda: df.groupby("region")["revenue"].sum().sum() > 0),
    ('df["region"].nunique() >= 2', lambda: df["region"].nunique() >= 2),
    ('df.dtypes["revenue"] == "float64"', lambda: df.dtypes["revenue"] == "float64"),
    ('not df.duplicated(subset=["order_id"]).any()', lambda: not df.duplicated(subset=["order_id"]).any()),
    ('len(df.groupby(["product","region"])) > 1', lambda: len(df.groupby(["product","region"])) > 1),
]

passed = 0
for name, fn in checks:
    try:
        result = fn()
        status = 'PASS' if result else 'FAIL'
        if result: passed += 1
    except Exception as e:
        status = f'ERROR: {e}'
    print(f'  {status}: {name}')

print(f'\n{passed}/{len(checks)} self-checks passed')
if passed == len(checks):
    print('\nAll good! Click "Run Tests" to submit for official validation.')
else:
    print('\nSome checks failed. Fix your code, then click "Run Tests".')

SELF-CHECK — OLTP vs OLAP
  PASS: len(df) >= 100
  PASS: "product" in df.columns
  PASS: "region" in df.columns
  PASS: df.groupby("region")["revenue"].sum().sum() > 0
  PASS: df["region"].nunique() >= 2
  PASS: df.dtypes["revenue"] == "float64"
  PASS: not df.duplicated(subset=["order_id"]).any()
  PASS: len(df.groupby(["product","region"])) > 1

8/8 self-checks passed

All good! Click "Run Tests" to submit for official validation.


In [ ]:
# Cell 5: Results Analysis
print('=' * 60)
print('RESULTS: OLTP vs OLAP and Data Storage Systems')
print('=' * 60)
print(f'Input rows: {len(df)}')
print(f'Processing complete')

# Display key metrics
print(f'\nRevenue by Region:')
print(df.groupby('region')['revenue'].sum().sort_values(ascending=False))

RESULTS: OLTP vs OLAP and Data Storage Systems
Input rows: 10000
Processing complete

Revenue by Region:
region
West     32704052.11
East     31929463.06
South    30902956.65
North    30756088.37
Name: revenue, dtype: float64


In [ ]:
# Cell 6: Self-Check — OLTP vs OLAP
# Run this cell to verify your work before clicking "Run Tests"
print('=' * 50)
print('SELF-CHECK — OLTP vs OLAP')
print('=' * 50)

checks = [
    ('len(df) >= 100', lambda: len(df) >= 100),
    ('"product" in df.columns', lambda: "product" in df.columns),
    ('"region" in df.columns', lambda: "region" in df.columns),
    ('df.groupby("region")["revenue"].sum().sum() > 0', lambda: df.groupby("region")["revenue"].sum().sum() > 0),
    ('df["region"].nunique() >= 2', lambda: df["region"].nunique() >= 2),
    ('df.dtypes["revenue"] == "float64"', lambda: df.dtypes["revenue"] == "float64"),
    ('not df.duplicated(subset=["order_id"]).any()', lambda: not df.duplicated(subset=["order_id"]).any()),
    ('len(df.groupby(["product","region"])) > 1', lambda: len(df.groupby(["product","region"])) > 1),
]

passed = 0
for name, fn in checks:
    try:
        result = fn()
        status = 'PASS' if result else 'FAIL'
        if result: passed += 1
    except Exception as e:
        status = f'ERROR: {e}'
    print(f'  {status}: {name}')

print(f'\n{passed}/{len(checks)} self-checks passed')
if passed == len(checks):
    print('\nAll good! Click "Run Tests" to submit for official validation.')
else:
    print('\nSome checks failed. Fix your code, then click "Run Tests".')

SELF-CHECK — OLTP vs OLAP
  PASS: len(df) >= 100
  PASS: "product" in df.columns
  PASS: "region" in df.columns
  PASS: df.groupby("region")["revenue"].sum().sum() > 0
  PASS: df["region"].nunique() >= 2
  PASS: df.dtypes["revenue"] == "float64"
  PASS: not df.duplicated(subset=["order_id"]).any()
  PASS: len(df.groupby(["product","region"])) > 1

8/8 self-checks passed

All good! Click "Run Tests" to submit for official validation.
